# Figure S7-S9: Batch-wise IQM distributions

Horizontal box plots of selected IQMs across `batch_device_software`, colored by scanner manufacturer, with one exported figure per IQM (`S7`, `S8`, `S9`).


In [19]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(fs)
  library(jsonlite)
  library(stringr)
  library(forcats)
  library(arrow)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) {
  stop("Could not locate config.json. Set CONFIG_PATH or run from within the project tree.")
}

config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

plot_style_file <- fs::path(project_root, "scripts", "utils", "plot_style.R")
if (!file.exists(plot_style_file)) stop("Missing plot style helper: ", plot_style_file)
source(plot_style_file)

plot_style <- get_plot_style(config)
font_family_use <- get_export_font_family()

fig_s7_dir <- fs::path(project_root, "figures", "Supplement", "FigureS7_S9")
fs::dir_create(fig_s7_dir, recurse = TRUE)


In [20]:
iqm_keep <- c("t1post_neighbor_corr", "t1post_dwi_contrast", "CNR0_median")
metric_labels <- c(
  t1post_neighbor_corr = "Preprocessed Neighboring dMRI Correlation (post-𝐵1)",
  t1post_dwi_contrast = "Preprocessed dMRI Contrast (post-𝐵1)",
  CNR0_median = "Median tNSR 𝑏=0"
)

harm_path <- fs::path(project_root, "data", "harmonized_data", "merged_data_meisler_analyses_harmonized.parquet")
if (!file.exists(harm_path)) stop("Missing harmonized data parquet: ", harm_path)

needed_cols <- c("scanner_manufacturer", "batch_device_software", iqm_keep)
schema_names <- arrow::open_dataset(harm_path, format = "parquet")$schema$names
missing_cols <- setdiff(needed_cols, schema_names)
if (length(missing_cols) > 0) {
  stop("Harmonized parquet missing required columns: ", paste(missing_cols, collapse = ", "))
}

vendor_order <- c("Siemens", "GE", "Philips")

batch_order <- arrow::read_parquet(harm_path, col_select = c("scanner_manufacturer", "batch_device_software")) %>%
  mutate(
    scanner_manufacturer = as.character(scanner_manufacturer),
    batch_device_software = as.character(batch_device_software)
  ) %>%
  filter(
    !is.na(scanner_manufacturer), scanner_manufacturer %in% vendor_order,
    !is.na(batch_device_software), nzchar(batch_device_software)
  ) %>%
  distinct(scanner_manufacturer, batch_device_software) %>%
  arrange(scanner_manufacturer, batch_device_software) %>%
  mutate(batch_id = row_number()) %>%
  pull(batch_device_software)

df_plot <- arrow::read_parquet(harm_path, col_select = needed_cols) %>%
  mutate(
    scanner_manufacturer = as.character(scanner_manufacturer),
    batch_device_software = as.character(batch_device_software)
  ) %>%
  filter(
    !is.na(scanner_manufacturer), scanner_manufacturer %in% vendor_order,
    !is.na(batch_device_software), nzchar(batch_device_software)
  ) %>%
  mutate(
    device_id = str_remove(batch_device_software, "\\..*$"),
    vendor_version = str_remove(batch_device_software, "^[^.]+\\."),
    software_version = str_remove(vendor_version, "^[^_]+_"),
    batch_label = paste0(device_id, "/", software_version)
  ) %>%
  pivot_longer(cols = all_of(iqm_keep), names_to = "iqm", values_to = "value") %>%
  mutate(
    value = suppressWarnings(as.numeric(value)),
    scanner_manufacturer = factor(scanner_manufacturer, levels = vendor_order),
    metric = factor(metric_labels[iqm], levels = unname(metric_labels))
  ) %>%
  filter(!is.na(value)) %>%
  group_by(scanner_manufacturer, batch_device_software, device_id, software_version, batch_label, metric) %>%
  mutate(batch_median = median(value, na.rm = TRUE)) %>%
  ungroup()


In [ ]:
metric_plan <- tibble::tibble(
  metric = factor(
    c(
      "Preprocessed Neighboring dMRI Correlation (post-𝐵1)",
      "Preprocessed dMRI Contrast (post-𝐵1)",
      "Median tNSR 𝑏=0"
    ),
    levels = levels(df_plot$metric)
  ),
  stub = c(
    "S7_preprocessed_dmri_correlation_post_b1",
    "S8_preprocessed_dmri_contrast_post_b1",
    "S9_median_tnsr_b0"
  )
)

for (idx in seq_len(nrow(metric_plan))) {
  metric_name <- as.character(metric_plan$metric[[idx]])
  metric_stub <- metric_plan$stub[[idx]]

  order_tbl <- df_plot %>%
    filter(metric == metric_name) %>%
    distinct(scanner_manufacturer, batch_label, device_id, software_version) %>%
    arrange(scanner_manufacturer, software_version, device_id) %>%
    group_by(scanner_manufacturer) %>%
    mutate(
      row_in_panel = row_number(),
      y_id = paste(scanner_manufacturer, batch_label, sep = "___")
    ) %>%
    ungroup()

  # global factor levels constructed in panel display order
  y_levels <- order_tbl %>%
    arrange(factor(scanner_manufacturer, levels = vendor_order), row_in_panel) %>%
    pull(y_id)

  order_tbl <- order_tbl %>%
    mutate(y_id = factor(y_id, levels = rev(y_levels)))

  df_metric <- df_plot %>%
    filter(metric == metric_name) %>%
    left_join(
      order_tbl %>%
        select(scanner_manufacturer, batch_label, device_id, software_version, row_in_panel, y_id),
      by = c("scanner_manufacturer", "batch_label", "device_id", "software_version")
    )

  # separator positions based on adjacent displayed rows within each vendor
  df_lines <- order_tbl %>%
    group_by(scanner_manufacturer) %>%
    arrange(row_in_panel, .by_group = TRUE) %>%
    mutate(
      n_vendor_rows = n(),
      next_software = lead(software_version),
      boundary_after = software_version != next_software
    ) %>%
    filter(boundary_after, !is.na(next_software)) %>%
    mutate(yintercept = n_vendor_rows - row_in_panel + 0.5) %>%
    ungroup()

  p_metric <- ggplot(
    df_metric,
    aes(x = value, y = y_id, fill = scanner_manufacturer)
  ) +
    geom_boxplot(width = 0.6, outlier.size = 0.2, linewidth = 0.2) +
    geom_hline(
      data = df_lines,
      aes(yintercept = yintercept),
      linetype = "dotted",
      color = "black",
      linewidth = 0.3
    ) +
    facet_wrap(~scanner_manufacturer, nrow = 1, scales = "free_y") +
    scale_y_discrete(labels = function(x) sub("^.*___", "", x)) +
    scale_fill_manual(
      values = c("Siemens" = "#1f77b4", "GE" = "#ff7f0e", "Philips" = "#2ca02c"),
      drop = FALSE
    ) +
    labs(
      x = metric_name,
      y = "Batch (device/software)",
      title = NULL
    ) +
    make_theme_pub(
      style = plot_style,
      legend_position = "none",
      axis_title_pt = 7,
      axis_text_pt = 7,
      plot_title_pt = 7,
      legend_title_pt = 7,
      legend_text_pt = 7,
      base_size_pt = 7
    ) +
    theme(
      text = element_text(family = font_family_use, size = 7),
      strip.text = element_text(size = 7),
      panel.grid.major.y = element_blank(),
      axis.text.y = element_text(size = 7)
    )

  print(p_metric)

  save_plot_outputs(
    plot_obj = p_metric,
    stub = metric_stub,
    out_dir = fig_s7_dir,
    width_in = 11,
    height_in = 10
  )
}

Warning message in geom_hline(data = df_lines, aes(yintercept = yintercept), linetype = "dotted", :
“Ignoring unknown parameters: `inherit.aes`”
